# Will This Visitor Buy? Purchase Intent Prediction
**NextGen Institute · Machine Learning Final Project**

## 1. Problem framing
- **Who has the problem:** an online store where most visitors leave without buying.
- **Decision the model supports:** show a coupon or a chat assistant only to visitors who are unlikely to buy on their own, instead of giving discounts to everyone.
- **Target:** `Revenue` (1 = the session ended in a purchase, 0 = no purchase).
- **Metric:** ROC-AUC to compare models, and **Recall / Precision of the buyer class** to choose the threshold. Accuracy is not used because the data is imbalanced (a model that always says "no purchase" would already look accurate).
- **Dataset:** Online Shoppers Purchasing Intention, UCI Machine Learning Repository: https://archive.ics.uci.edu/dataset/468

> **Colab:** run the next cell once to install the dataset downloader.

In [ ]:
!pip install -q ucimlrepo

In [ ]:
import json, joblib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, precision_score, recall_score, f1_score,
                             average_precision_score, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
RANDOM_STATE = 42

## 2. Data understanding

In [ ]:
from ucimlrepo import fetch_ucirepo
ds = fetch_ucirepo(id=468)
df = pd.concat([ds.data.features, ds.data.targets], axis=1)

# If the download fails, upload the CSV manually and use this instead:
# df = pd.read_csv("online_shoppers_intention.csv")

print("Rows, columns:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0] if df.isna().sum().sum() else "None")
print("\nDuplicate rows:", df.duplicated().sum())
print("\nClass balance:")
print(df["Revenue"].value_counts())
print(df["Revenue"].value_counts(normalize=True).round(3))

In [ ]:
df.describe().T

**Column dictionary**

| Column | Meaning |
|---|---|
| `Administrative`, `Informational`, `ProductRelated` | Number of pages visited of each type in the session |
| `*_Duration` | Seconds spent on each page type |
| `BounceRates` | Share of visitors who leave after one page (Google Analytics) |
| `ExitRates` | Share of views of a page that were the last in the session |
| `PageValues` | Average value of the pages visited before a purchase |
| `SpecialDay` | Closeness of the visit to a special day (e.g. Valentine's), 0 to 1 |
| `Month`, `Weekend` | When the visit happened |
| `OperatingSystems`, `Browser`, `Region`, `TrafficType` | Technical / geographic IDs (categorical codes) |
| `VisitorType` | New, Returning or Other |
| `Revenue` | **Target**: purchase or not |

The dataset has no missing values, a small number of duplicate rows (removed in preprocessing) and about 15% buyers.

## 3. Exploratory data analysis (EDA)
### Univariate

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(x="Revenue", data=df, ax=ax[0]); ax[0].set_title("Target balance")
df["Revenue"].value_counts(normalize=True).plot.pie(autopct="%1.1f%%", ax=ax[1]); ax[1].set_ylabel("")
plt.show()

num_plot = ["Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
            "ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates", "PageValues", "SpecialDay"]
df[num_plot].hist(bins=30, figsize=(14, 9))
plt.tight_layout(); plt.show()

### Bivariate (features vs. the target)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, c in zip(ax, ["PageValues", "ExitRates", "BounceRates"]):
    sns.boxplot(x="Revenue", y=c, data=df, ax=a, showfliers=False)
    a.set_title(c + " by Revenue")
plt.tight_layout(); plt.show()

In [ ]:
order = ["Feb", "Mar", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.barplot(x="Month", y="Revenue", data=df, order=order, ax=ax[0]); ax[0].set_title("Conversion rate by month")
sns.barplot(x="VisitorType", y="Revenue", data=df, ax=ax[1]); ax[1].set_title("Conversion rate by visitor type")
sns.barplot(x="Weekend", y="Revenue", data=df, ax=ax[2]); ax[2].set_title("Conversion rate: weekend vs weekday")
plt.tight_layout(); plt.show()

### Multivariate

In [ ]:
corr = df.select_dtypes("number").assign(Revenue=df["Revenue"].astype(int)).corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation heatmap"); plt.show()
corr["Revenue"].drop("Revenue").sort_values(ascending=False).round(3)

In [ ]:
# Numbers to support your written insights
print("Overall conversion rate: %.1f%%" % (100 * df["Revenue"].mean()))
print("\nConversion by visitor type (%):")
print((100 * df.groupby("VisitorType")["Revenue"].mean()).round(1))
print("\nConversion by month (%):")
print((100 * df.groupby("Month")["Revenue"].mean()).reindex(order).round(1))
print("\nConversion by whether PageValues > 0 (%):")
print((100 * df.groupby(df["PageValues"] > 0)["Revenue"].mean()).round(1))

### Insights (write 3 to 5 sentences using the numbers printed above)
1. *Example: Only about X% of sessions end in a purchase, so the data is imbalanced and accuracy would be misleading.*
2. *`PageValues` is the strongest signal: sessions with PageValues > 0 convert at X% versus Y% for the rest.*
3. *Returning / New visitors convert at ...*
4. *Conversion peaks in the month of ...*
5. *Higher ExitRates / BounceRates go with lower purchase probability because ...*

## 4. Preprocessing and Pipeline
- Remove duplicate rows.
- Convert `Weekend` to 0/1 and treat the ID-like columns (`Month`, `VisitorType`, `OperatingSystems`, `Browser`, `Region`, `TrafficType`) as **categorical**.
- **Split first** (80% train / 20% test, stratified), then fit scaling and encoding **inside a scikit-learn Pipeline** so nothing is learned from the test set (no leakage).
- Numeric features are scaled; categorical features are one-hot encoded (`handle_unknown="ignore"`).

In [ ]:
data = df.drop_duplicates().reset_index(drop=True).copy()
data["Revenue"] = data["Revenue"].astype(int)
data["Weekend"] = data["Weekend"].astype(int)
print("Rows after removing duplicates:", len(data))

y = data["Revenue"]
X = data.drop(columns="Revenue")

cat_cols = ["Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType"]
num_cols = [c for c in X.columns if c not in cat_cols]
X[cat_cols] = X[cat_cols].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Buyers in train: %.1f%% | in test: %.1f%%" % (100 * y_train.mean(), 100 * y_test.mean()))

def make_pipe(model):
    pre = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    return Pipeline([("pre", pre), ("model", model)])

## 5. Modeling and tuning
- **Baseline:** always predict the majority class ("no purchase").
- **Three algorithms:** Logistic Regression, Random Forest, Gradient Boosting.
- **Tuning:** `RandomizedSearchCV` with 3-fold stratified cross-validation, scored by ROC-AUC, **on the training set only**. Class imbalance is handled with `class_weight="balanced"` where available.

In [ ]:
candidates = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, class_weight="balanced"),
        {"model__C": [0.01, 0.1, 1, 10]}),
    "Random Forest": (
        RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        {"model__n_estimators": [100, 200], "model__max_depth": [5, 10, 20, None],
         "model__min_samples_leaf": [1, 3, 5]}),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {"model__n_estimators": [100, 200, 300], "model__learning_rate": [0.03, 0.1],
         "model__max_depth": [2, 3, 4]}),
}
cv = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE)

def scores(name, cv_auc, p, threshold=0.5):
    pred = (p >= threshold).astype(int)
    return {"Model": name, "CV ROC-AUC": cv_auc,
            "Test ROC-AUC": roc_auc_score(y_test, p),
            "Precision": precision_score(y_test, pred, zero_division=0),
            "Recall": recall_score(y_test, pred),
            "F1": f1_score(y_test, pred),
            "PR-AUC": average_precision_score(y_test, p)}

rows, fitted, best_params = [], {}, {}

base = make_pipe(DummyClassifier(strategy="most_frequent")).fit(X_train, y_train)
rows.append(scores("Baseline (majority class)", np.nan, base.predict_proba(X_test)[:, 1]))

for name, (model, grid) in candidates.items():
    print("Tuning", name, "...")
    n_iter = min(4, int(np.prod([len(v) for v in grid.values()])))
    search = RandomizedSearchCV(make_pipe(model), grid, n_iter=n_iter, scoring="roc_auc",
                                cv=cv, random_state=RANDOM_STATE, n_jobs=-1)
    search.fit(X_train, y_train)
    fitted[name] = (search.best_estimator_, search.best_score_)
    best_params[name] = search.best_params_
    rows.append(scores(name, search.best_score_, search.best_estimator_.predict_proba(X_test)[:, 1]))

results = pd.DataFrame(rows).round(3)
results

In [ ]:
for k, v in best_params.items():
    print(k, "->", v)

## 6. Evaluation on the untouched test set
The best model is chosen by **cross-validation score on the training data** (not by looking at the test set).

In [ ]:
best_name = max(fitted, key=lambda k: fitted[k][1])
best_model = fitted[best_name][0]
p_test = best_model.predict_proba(X_test)[:, 1]
print("Best model:", best_name)
print("\nClassification report at threshold 0.5:\n")
print(classification_report(y_test, (p_test >= 0.5).astype(int), target_names=["No purchase", "Purchase"]))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test, ax=ax[0], colorbar=False)
ax[0].set_title("Confusion matrix (threshold 0.5)")
for n, (m, _) in fitted.items():
    RocCurveDisplay.from_estimator(m, X_test, y_test, ax=ax[1], name=n)
ax[1].set_title("ROC curves (test set)")
plt.tight_layout(); plt.show()

### Choosing the decision threshold
The default 0.5 is not necessarily the best business choice. A **lower threshold** catches more buyers (higher recall) but flags more non-buyers (lower precision). The table below lets you pick a threshold and justify it.

In [ ]:
thr = pd.DataFrame({
    "Threshold": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6],
})
thr["Precision"] = [precision_score(y_test, (p_test >= t).astype(int), zero_division=0) for t in thr["Threshold"]]
thr["Recall"] = [recall_score(y_test, (p_test >= t).astype(int)) for t in thr["Threshold"]]
thr["Flagged as likely buyer (%)"] = [100 * (p_test >= t).mean() for t in thr["Threshold"]]
thr.round(3)

### Error analysis

In [ ]:
err = X_test.assign(y=y_test, p=p_test)
fn = err[(err.y == 1) & (err.p < 0.5)]
fp = err[(err.y == 0) & (err.p >= 0.5)]
print("Missed buyers (false negatives):", len(fn), "of", int(y_test.sum()))
print("False alarms (false positives): ", len(fp), "of", int((y_test == 0).sum()))

print("\nMissed buyers: key features")
print(fn[["PageValues", "ExitRates", "BounceRates"]].describe().round(3))
print("\nCorrectly caught buyers: key features")
print(err[(err.y == 1) & (err.p >= 0.5)][["PageValues", "ExitRates", "BounceRates"]].describe().round(3))

### Which features matter most? (permutation importance on the test set)

In [ ]:
imp = permutation_importance(best_model, X_test, y_test, scoring="roc_auc",
                             n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
imp_df = (pd.DataFrame({"feature": X_test.columns, "importance": imp.importances_mean})
          .sort_values("importance", ascending=False).head(10))
plt.figure(figsize=(7, 4))
sns.barplot(x="importance", y="feature", data=imp_df, color="#2563eb")
plt.title("Top 10 features (drop in ROC-AUC when shuffled)")
plt.tight_layout(); plt.show()

## 7. Business insight and recommendations
*Fill this section with your own numbers from the outputs above.*

**What the results mean**
- The best model (**[model name]**) reaches a test ROC-AUC of **[X]**, far above the baseline (0.50), so browsing behavior really does predict purchases.
- At threshold 0.5 it catches **[X]%** of buyers with **[Y]%** precision. At threshold **[chosen value]** it catches **[X]%** of buyers with **[Y]%** precision.
- The most important signals are **[top features]**.

**What the store should do**
1. Visitors with a **high** purchase probability: show nothing (no discount needed).
2. Visitors in the **middle** range: show a coupon.
3. Visitors with a **low** probability but who are still browsing: offer the chat assistant.
4. Choose the thresholds from the table above based on the cost of a coupon versus the value of a sale.

**Limitations**
- Data comes from one store and one period, so the model may not generalize to other stores or seasons.
- `PageValues` is computed from past behavior and must be available in real time for the model to work in production.
- Buyers with no browsing history are hard for the model to catch (see the error analysis).
- The model estimates probability. It does not prove that a coupon *causes* a purchase; that needs an A/B test.

## 8. Save the pipeline for the web app

In [ ]:
joblib.dump(best_model, "model.joblib")
defaults = {c: float(X_train[c].median()) for c in num_cols}
defaults.update({c: X_train[c].mode()[0] for c in cat_cols})
json.dump({"defaults": defaults, "columns": list(X.columns), "best_model": best_name},
          open("app_config.json", "w"), indent=2)
results.to_csv("results.csv", index=False)
print("Saved: model.joblib, app_config.json, results.csv")

# Colab only: uncomment to download the files to your computer
# from google.colab import files
# for f in ["model.joblib", "app_config.json", "results.csv"]:
#     files.download(f)

**Dataset credit:** Sakar, C. and Kastro, Y. (2018). *Online Shoppers Purchasing Intention Dataset*. UCI Machine Learning Repository. https://archive.ics.uci.edu/dataset/468